# 01 EDA - Bahn Delay Story

Objective: profile the monthly processed Deutsche Bahn stop data before making claims about delays.

Questions for this notebook:
- Which months, stations, and train types are covered?
- How are delay minutes distributed?
- Where do cancellations and severe delays concentrate?


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))

from bahn_delay_story.config import source_parquet_files
from bahn_delay_story.pipeline import duckdb_path_list

pd.set_option("display.max_columns", 50)
ROOT


## Data Availability

Expected source files live in `data/raw/yearly_processed_data/` as `data-YYYY-MM.parquet`. The alternate `monthly_processed_data/` folder is also supported.

In [ ]:
files = source_parquet_files()
source_path_sql = duckdb_path_list(files)
print(f"{len(files)} monthly files found")

if files:
    print(files[0].name, "->", files[-1].name)
else:
    print("Download data first. See README.md for the Hugging Face command.")


## Coverage Profile

In [ ]:
if files:
    with duckdb.connect() as con:
        con.execute(
            f"""
            CREATE OR REPLACE VIEW stops_raw AS
            SELECT *
            FROM read_parquet({source_path_sql}, union_by_name = true)
            """,
        )

        coverage = con.execute(
            """
            SELECT
              DATE_TRUNC('month', time)::DATE AS month,
              COUNT(*) AS rows,
              COUNT(DISTINCT station_name) AS stations,
              COUNT(DISTINCT train_type) AS train_types,
              MIN(time) AS min_time,
              MAX(time) AS max_time
            FROM stops_raw
            GROUP BY 1
            ORDER BY 1
            """
        ).df()
else:
    coverage = pd.DataFrame()

coverage


## Train-Type Delay Summary

In [ ]:
if files:
    with duckdb.connect() as con:
        con.execute(f"CREATE OR REPLACE VIEW stops_raw AS SELECT * FROM read_parquet({source_path_sql}, union_by_name = true)")
        train_type_summary = con.execute(
            """
            SELECT
              UPPER(train_type) AS train_type,
              COUNT(*) AS stops,
              AVG(delay_in_min) AS avg_delay_min,
              MEDIAN(delay_in_min) AS median_delay_min,
              QUANTILE_CONT(delay_in_min, 0.9) AS p90_delay_min,
              AVG(CASE WHEN delay_in_min >= 6 AND NOT COALESCE(is_canceled, false) THEN 1.0 ELSE 0.0 END) AS late_share_6_min,
              AVG(CASE WHEN COALESCE(is_canceled, false) THEN 1.0 ELSE 0.0 END) AS cancellation_share
            FROM stops_raw
            GROUP BY 1
            HAVING COUNT(*) >= 1000
            ORDER BY late_share_6_min DESC
            """
        ).df()
else:
    train_type_summary = pd.DataFrame()

train_type_summary.head(20)


## Station Outliers

In [ ]:
if files:
    with duckdb.connect() as con:
        con.execute(f"CREATE OR REPLACE VIEW stops_raw AS SELECT * FROM read_parquet({source_path_sql}, union_by_name = true)")
        station_summary = con.execute(
            """
            SELECT
              station_name,
              COUNT(*) AS stops,
              COUNT(DISTINCT train_type) AS train_types,
              AVG(delay_in_min) AS avg_delay_min,
              QUANTILE_CONT(delay_in_min, 0.9) AS p90_delay_min,
              AVG(CASE WHEN delay_in_min >= 6 AND NOT COALESCE(is_canceled, false) THEN 1.0 ELSE 0.0 END) AS late_share_6_min
            FROM stops_raw
            GROUP BY 1
            HAVING COUNT(*) >= 5000
            ORDER BY late_share_6_min DESC
            LIMIT 30
            """
        ).df()
else:
    station_summary = pd.DataFrame()

station_summary


## EDA Notes

- Document any coverage jumps before building trend charts.
- Decide whether the main story uses all stops or a stable station subset.
- Save surprising station/train-type patterns for `02_analysis.ipynb`.